# 04 — ML Modelling
Train XGBoost per the project requirements.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from xgboost import XGBRegressor

ROOT = Path().resolve().parents[0]
df = pd.read_parquet(ROOT / 'data/processed/features.parquet')

In [2]:
# time based split: train on 2022, test on 2025
train = df[df['Start'].dt.year == 2022]
test = df[df['Start'].dt.year == 2025]

# if dates are different, fallback to 80/20 split
if train.empty or test.empty:
    split_idx = int(len(df) * 0.8)
    train = df.iloc[:split_idx]
    test = df.iloc[split_idx:]

features = ['hour', 'dayofweek', 'month', 'is_weekend', 'PM25_lag1', 'PM25_lag24', 'PM25_roll24', 
            'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
            
target = 'target_PM25_24h'

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]
print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

Train samples: 17424, Test samples: 17417


In [3]:
# Train XGBoost
model = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
rmse = root_mean_squared_error(y_test, preds)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

MAE: 7.74
RMSE: 10.48


In [4]:
# Save model
models_dir = ROOT / 'models'
models_dir.mkdir(exist_ok=True)

import pickle
with open(models_dir / 'model.pkl', 'wb') as f:
    pickle.dump(model, f)
    
meta = {"features": features, "mae": float(mae), "rmse": float(rmse)}
with open(models_dir / 'model_metadata.json', 'w') as f:
    json.dump(meta, f)
    
print("Model and metadata saved.")

Model and metadata saved.
